# MiraeVaani 2.0 — Colab #1: STT Service

Runs **Faster-Whisper large-v3** for Indian language speech-to-text.

- Supports: Hindi, Tamil, Telugu, Marathi, Bengali, Gujarati, Kannada, Malayalam, Punjabi, Urdu, English
- Auto-detects language per utterance
- Exposes `POST /transcribe` via ngrok

**Setup:** Runtime → Change runtime type → **T4 GPU**

In [1]:
# Cell 1: Verify GPU
import subprocess, torch
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip())
print('CUDA:', torch.cuda.is_available())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: Tesla T4, 15360 MiB
CUDA: True
VRAM: 15.6 GB


In [2]:
# Cell 2: Install dependencies
!pip install -q faster-whisper
!pip install -q fastapi uvicorn python-multipart httpx pyngrok
print('✅ Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.5 MB/s eta 0:00:00
✅ Done


In [3]:
# Cell 3: Configure ngrok
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3Fofu4RyRZPuzISF7Qfd1WR6AQQ_4ejpURdwsv3CpxuyTPutV"  # https://dashboard.ngrok.com
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('✅ ngrok configured')

✅ ngrok configured


In [4]:
# Cell 4: Write STT service
stt_code = '''
import os
import tempfile
import logging
from typing import Optional
from fastapi import FastAPI, UploadFile, File, Form
from faster_whisper import WhisperModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="MiraeVaani STT")

logger.info("Loading Faster-Whisper large-v3...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
logger.info("STT model loaded")


@app.get("/health")
def health():
    return {"status": "ok", "model": "faster-whisper-large-v3"}


@app.post("/transcribe")
async def transcribe(
    file: UploadFile = File(...),
    language: Optional[str] = Form(default=None),
):
    audio_bytes = await file.read()
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio_bytes)
        tmp_path = f.name
    try:
        lang_arg = language if (language and language != "auto") else None
        segments, info = model.transcribe(
            tmp_path,
            language=lang_arg,
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=300, speech_pad_ms=200),
        )
        transcript = " ".join(s.text.strip() for s in segments).strip()
    finally:
        os.unlink(tmp_path)
    logger.info("[%s %.0f%%] %s", info.language, info.language_probability * 100, transcript)
    return {
        "transcript": transcript,
        "language": info.language,
        "language_probability": round(info.language_probability, 3),
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''

with open('/content/stt_service.py', 'w') as f:
    f.write(stt_code)
print('✅ stt_service.py written')

✅ stt_service.py written


In [5]:
# Cell 5: Start STT service + expose via ngrok
import subprocess, time, httpx
from pyngrok import ngrok

log = open('/content/stt.log', 'w')
proc = subprocess.Popen(['python', '/content/stt_service.py'], stdout=log, stderr=log)
print('Loading Whisper large-v3 (~2 GB, ~60s)...')

for i in range(24):
    time.sleep(5)
    if proc.poll() is not None:
        log.flush()
        print('❌ Process crashed:')
        print(open('/content/stt.log').read()[-2000:])
        break
    try:
        r = httpx.get('http://localhost:8001/health', timeout=3)
        if r.status_code == 200:
            print(f'✅ STT service ready ({(i+1)*5}s)')
            break
    except Exception:
        if i % 4 == 3:
            print(f'  Still loading... ({(i+1)*5}s)')
else:
    print('⚠️  Timed out — check /content/stt.log')

tunnel = ngrok.connect(8001, 'http')
print()
print('=' * 60)
print('  PASTE THIS INTO YOUR LAPTOP .env')
print('=' * 60)
print(f'STT_BASE_URL={tunnel.public_url}')
print('=' * 60)

Loading Whisper large-v3 (~2 GB, ~60s)...
  Still loading... (20s)
  Still loading... (40s)
✅ STT service ready (50s)

  PASTE THIS INTO YOUR LAPTOP .env
STT_BASE_URL=https://plenty-undermine-musky.ngrok-free.dev


In [ ]:
# Cell 6: Keep-alive (keep running while calls are active)
import time, httpx
print('STT keep-alive running...')
i = 0
while True:
    try:
        ok = httpx.get('http://localhost:8001/health', timeout=3).status_code == 200
        i += 1
        if i % 20 == 0:
            print(f'[{i*30}s] STT alive={ok}')
    except Exception as e:
        print(f'⚠️  {e}')
    time.sleep(30)

STT keep-alive running...
[600s] STT alive=True
